# Batch Preprocessing

**Author:** Noah Mba, noah.mba@fu-berlin.de  
**Date:** July 13, 2026  
**AI Acknowledgements:** Co-authored/Supported by Gemini and Claude 3.5 Sonnet  

---

This notebook can run the preprocessing pipeline for all subjects that should be analyzed. It does so by defining a preprocessing function that takes either a single or multiple subject IDs as inputs.

### 1. Setup: Loading modules and objects, creating basic paths and helper function

This initialization cell prepares the workspace by loading the relevant modules and specifying the relevant paths and directories.

In [ ]:
# ==========================================
# 1. IMPORTS & GLOBAL PATHS
# ==========================================
import mne
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
from autoreject import AutoReject
import matplotlib
matplotlib.use('Agg')  # No interactive popups during batch runs
import matplotlib.pyplot as plt
mne.viz.set_browser_backend("matplotlib")

# ==========================================
# 2. Path Definitions & Global Variables 
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"
config_path = derivatives_dir / "preprocessing_config.csv"

# ==========================================
# 3. Load Configuration dataframe
# ==========================================
config_df = pd.read_csv(config_path)

# ==========================================
# 3. HELPER FUNCTION: SAVE A FIGURE SAFELY
# ==========================================
def save_fig(fig, out_dir, fname):
    """Save and close a figure, creating the directory if needed."""
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / fname, dpi=100, bbox_inches='tight')
    plt.close(fig)

Using matplotlib as 2D backend.


### 3. Define the Subject-Level Processing Function

Automated preprocessing: channel interpolation → filtering → downsampling → preliminary epoching → ICA fit → component identification/removal → final epoching → merge behavioral metadata → Autoreject → baseline → re-reference.

In [ ]:
# ==========================================
# 2. MAIN PREPROCESSING FUNCTION
# ==========================================
def preprocess_subject(
    subject_id,
    bad_channels,
    bad_ics,
    l_freq=0.1,
    h_freq=40.0,
    resample_sfreq=250.0,
    epoch_tmin=-0.5,
    epoch_tmax=1.0,
    baseline_corr=(None, 0),
    overwrite=False,
):
    """
    Run the full preprocessing pipeline for one subject.

    Parameters
    ----------
    subject_id : str
        BIDS subject label, e.g. "23".
    bad_channels : list of str
        Channels to mark as bad and interpolate, from notebook 2's QC.
    bad_ics : list of int
        ICA component indices to exclude, from notebook 2's QC.
    overwrite : bool
        If False and output files already exist, skip this subject
        (idempotent re-runs — safe to re-execute the batch loop).

    Returns
    -------
    dict
        Summary log (also saved as JSON) with key metrics for this subject.
    """
    subj_deriv_dir = derivatives_dir / f"sub-{subject_id}" / "eeg"
    plots_dir = subj_deriv_dir / "qc_plots"
    log_path = subj_deriv_dir / f"sub-{subject_id}_preproc_log.json"
    beh_path = bids_root / f"sub-{subject_id}" / "beh" / f"sub-{subject_id}_task-loc_label-merged_beh.csv"
    epochs_enc_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-encoding_epo.fif"
    epochs_ret_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-retrieval_epo.fif"

    # --- Idempotency check: skip if already done ---
    if not overwrite and epochs_enc_path.exists() and epochs_ret_path.exists():
        print(f"sub-{subject_id}: outputs already exist, skipping (overwrite=False).")
        with open(log_path, 'r') as f:
            return json.load(f)

    log = {"subject": subject_id, "steps_completed": [], "warnings": []}

    # ==========================================
    # STEP 1: LOAD RAW + APPLY BAD CHANNELS
    # ==========================================
    bids_path = BIDSPath(subject=subject_id, task='loc', datatype='eeg', root=bids_root)
    raw = read_raw_bids(bids_path=bids_path, verbose='error')
    raw.load_data()
    raw.info['bads'] = bad_channels
    log["bad_channels"] = bad_channels
    log["steps_completed"].append("load_raw")

    # Butterfly plot BEFORE filtering
    fig_before = raw.copy().pick('eeg').plot(
        duration=10, n_channels=len(raw.ch_names), show=False,
        scalings=dict(eeg=20e-6), butterfly=True,
        theme='light'  # unrelated, but explicit is nice
    )
    save_fig(fig_before, plots_dir, "01_butterfly_before_filter.png")

    # ==========================================
    # STEP 2: FILTERING (low, high)
    # ==========================================
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    log["filter_params"] = {"l_freq": l_freq, "h_freq": h_freq}
    log["steps_completed"].append("filter")

    # Butterfly plot AFTER filtering
    fig_after = raw.copy().pick('eeg').plot(
        duration=10, n_channels=len(raw.ch_names), show=False,
        scalings=dict(eeg=20e-6), butterfly=True
    )
    save_fig(fig_after, plots_dir, "02_butterfly_after_filter.png")

    # Interpolate bad channels now (before ICA apply / epoching)
    if bad_channels:
        raw.interpolate_bads(reset_bads=True, verbose=False)
        log["steps_completed"].append("interpolate_bads")

    # ==========================================
    # STEP 3: DOWNSAMPLE
    # ==========================================
    raw.resample(resample_sfreq, verbose=False)
    log["resample_sfreq"] = resample_sfreq
    log["steps_completed"].append("resample")

    # ==========================================
    # STEP 4: APPLY ICA (using bad_ics from notebook 2)
    # ==========================================
    # #ICA.apply() needs the actual unmixing matrix, which was saved in the previous notebook
    ica_path = derivatives_dir / f"sub-{subject_id}" / "eeg" / "ica_qc" / f"sub-{subject_id}_ica.fif"
    if not ica_path.exists():
        raise FileNotFoundError(
            f"No saved ICA solution found for sub-{subject_id} at {ica_path}. "
            f"Save ica.save(...) at the end of notebook 2."
        )
    ica = mne.preprocessing.read_ica(ica_path)
    ica.exclude = bad_ics
    ica.apply(raw)
    log["excluded_ics"] = bad_ics
    log["steps_completed"].append("ica_apply")
   
    # ==========================================
    # STEP 5: EPOCHING (encoding + retrieval, separately)
    # ==========================================

    # --- Load and prepare behavioral data ---
    beh_path = (bids_root / f"sub-{subject_id}" / "beh"
                / f"sub-{subject_id}_task-loc_label-merged_beh.csv")
    beh_df = pd.read_csv(beh_path)

    def sc_si_pc_pi(row):
        sc_si = 'SI' if row['enc_high_prediction'] == 1 else 'SC'
        pc_pi = 'PI' if row['enc_low_prediction'] == 1 else 'PC'
        return sc_si, pc_pi

    # Encoding: the 64 rows that were actually encoded, in presentation order
    beh_enc = (beh_df.dropna(subset=['enc_trial_count'])
                      .sort_values('enc_trial_count')
                      .reset_index(drop=True))
    beh_enc['condition_label'] = beh_enc.apply(
        lambda r: 'Target/' + '/'.join(sc_si_pc_pi(r)), axis=1
    )

    # Retrieval: all 84 rows, in their retrieval presentation order
    beh_ret = beh_df.sort_values('ret_trial_count').reset_index(drop=True)
    def ret_label(row):
        if row['ret_trial_type'] == 'new':
            return 'Cue/New'
        return 'Cue/Old/' + '/'.join(sc_si_pc_pi(row))
    beh_ret['condition_label'] = beh_ret.apply(ret_label, axis=1)

    beh_metadata = {'encoding': beh_enc, 'retrieval': beh_ret}

    # --- Extract events from annotations ---
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    present_ids = set(events[:, 2])

    enc_target_event_ids = {
        'Target/SC/PC': event_id['tgt_sc_pc'],
        'Target/SC/PI': event_id['tgt_sc_pi'],
        'Target/SI/PC': event_id['tgt_si_pc'],
        'Target/SI/PI': event_id['tgt_si_pi'],
    }

    ret_cue_event_ids = {
        'Cue/Old/SC/PC': event_id['ret_cue_old_sc_pc'],
        'Cue/Old/SC/PI': event_id['ret_cue_old_sc_pi'],
        'Cue/Old/SI/PC': event_id['ret_cue_old_si_pc'],
        'Cue/Old/SI/PI': event_id['ret_cue_old_si_pi'],
        'Cue/New': event_id['ret_cue_new'],
    }

    epochs_dict = {}
    for phase, phase_event_ids in [('encoding', enc_target_event_ids),
                                    ('retrieval', ret_cue_event_ids)]:

        available = {label: eid for label, eid in phase_event_ids.items()
                     if eid in present_ids}
        missing = set(phase_event_ids) - set(available)
        if missing:
            log["warnings"].append(
                f"{phase}: missing conditions in recording — {sorted(missing)}"
            )
        if not available:
            log["warnings"].append(f"No events found for {phase} phase — skipped.")
            continue

        # Filter the full events array down to just this phase's matching triggers,
        # in chronological order — this is what metadata needs to align against.
        phase_mask = np.isin(events[:, 2], list(available.values()))
        phase_events = events[phase_mask]

        metadata = beh_metadata[phase]

        if len(metadata) != len(phase_events):
            log["warnings"].append(
                f"{phase}: behavioral rows ({len(metadata)}) != triggered EEG events "
                f"({len(phase_events)}) — metadata NOT attached for sub-{subject_id}, check alignment."
            )
            metadata_to_use = None
        else:
            metadata_to_use = metadata

        epochs = mne.Epochs(
            raw, phase_events, event_id=available,
            tmin=epoch_tmin, tmax=epoch_tmax, baseline=baseline_corr,
            preload=True, reject_by_annotation=True, verbose=False,
            metadata=metadata_to_use,
        )

        if metadata_to_use is not None:
            id_to_label = {v: k for k, v in available.items()}
            triggered_labels = [id_to_label[code] for code in epochs.events[:, 2]]
            beh_labels = epochs.metadata['condition_label'].tolist()
            mismatches = [i for i, (a, b) in enumerate(zip(triggered_labels, beh_labels)) if a != b]
            if mismatches:
                log["warnings"].append(
                    f"{phase}: {len(mismatches)} trigger/CSV condition mismatches "
                    f"(first at epoch index {mismatches[0]}) — inspect trial order."
                )

        epochs_dict[phase] = epochs

    log["steps_completed"].append("epoching")
    log["n_epochs_pre_autoreject"] = {k: len(v) for k, v in epochs_dict.items()}
    # ==========================================
    # STEP 6: AUTOREJECT (separately per phase)
    # ==========================================
    epochs_clean = {}
    for phase, epochs in epochs_dict.items():
        # Evoked BEFORE autoreject
        fig_evoked_before = epochs.average().plot(show=False)
        save_fig(fig_evoked_before, plots_dir, f"03_evoked_{phase}_before_autoreject.png")

        ar = AutoReject(random_state=97, n_jobs=1, verbose=False)
        epochs_ar, reject_log = ar.fit_transform(epochs, return_log=True)
        epochs_clean[phase] = epochs_ar

        log.setdefault("n_epochs_post_autoreject", {})[phase] = len(epochs_ar)
        log.setdefault("autoreject_pct_dropped", {})[phase] = round(
            100 * (1 - len(epochs_ar) / len(epochs)), 1
        )

        # Evoked AFTER autoreject
        fig_evoked_after = epochs_ar.average().plot(show=False)
        save_fig(fig_evoked_after, plots_dir, f"04_evoked_{phase}_after_autoreject.png")

        # Reject log visualization (which epochs/channels were dropped/interpolated)
        fig_reject = reject_log.plot(show=False)
        save_fig(fig_reject, plots_dir, f"05_autoreject_log_{phase}.png")

    log["steps_completed"].append("autoreject")

    # ==========================================
    # STEP 7: AVERAGE RE-REFERENCING
    # ==========================================
    for phase, epochs in epochs_clean.items():
        epochs.set_eeg_reference('average', projection=False, verbose=False)
    log["steps_completed"].append("average_reference")

    # ==========================================
    # STEP 8: SAVE OUTPUTS
    # ==========================================
    subj_deriv_dir.mkdir(parents=True, exist_ok=True)
    epochs_clean['encoding'].save(epochs_enc_path, overwrite=overwrite) if 'encoding' in epochs_clean else None
    epochs_clean['retrieval'].save(epochs_ret_path, overwrite=overwrite) if 'retrieval' in epochs_clean else None

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"sub-{subject_id}: preprocessing complete. "
          f"Encoding epochs: {log.get('n_epochs_post_autoreject', {}).get('encoding', 'N/A')}, "
          f"Retrieval epochs: {log.get('n_epochs_post_autoreject', {}).get('retrieval', 'N/A')}")

    return log

**Option 1: Test Pipeline on A Single Subject**

In [10]:
# ==========================================
# 3. SINGLE-SUBJECT TEST RUN
# ==========================================
# Test on one subject before running the full batch
test_log = preprocess_subject(
    subject_id="17",
    bad_channels=["FC4", "P2"],
    bad_ics=[],
    overwrite=True,
)

Reading 0 ... 3611599  =      0.000 ...  3611.599 secs...
Reading c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-17\eeg\ica_qc\sub-17_ica.fif ...
Now restoring ICA solution ...
Ready.
Applying ICA to Raw instance
    Transforming to ICA space (60 components)
    Zeroing out 0 ICA components
    Projecting back using 61 PCA components
Dropped 1 epoch: 27
Dropped 22 epochs: 1, 2, 6, 13, 15, 19, 20, 24, 26, 29, 39, 42, 43, 52, 57, 60, 62, 68, 71, 76, 81, 82
Applying baseline correction (mode: mean)
Applying baseline correction (mode: mean)
sub-17: preprocessing complete. Encoding epochs: 63, Retrieval epochs: 62


**Option 2: Run Pipeline for all subjects**

In [12]:
# ==========================================
# 4. BATCH RUN OVER ALL MANUALLY-CHECKED SUBJECTS
# ==========================================
df_config = pd.read_csv(config_path, dtype={'subject': str})

# Only process subjects that are:
# - not behaviorally excluded
# - have complete EEG recordings for both phases (adjust if partial data is acceptable)
ready_mask = (
    (~df_config['is_excluded'].astype(bool)) &
    (df_config['eeg_enc_recorded'] == True) &
    (df_config['eeg_ret_recorded'] == True)
)
subjects_to_process = df_config[ready_mask]

print(f"{len(subjects_to_process)} of {len(df_config)} subjects ready for batch preprocessing.")

batch_summary = []
for _, row in subjects_to_process.iterrows():
    subj_id = row['subject']
    bad_chs = [c.strip() for c in row['bad_channels'].split(',')] if pd.notna(row['bad_channels']) and row['bad_channels'] else []
    bad_ic_list = [int(x.strip()) for x in row['bad_icas'].split(',')] if pd.notna(row['bad_icas']) and row['bad_icas'] else []

    try:
        log = preprocess_subject(
            subject_id=subj_id,
            bad_channels=bad_chs,
            bad_ics=bad_ic_list,
            overwrite=False,  # skip subjects already processed
        )
        batch_summary.append({"subject": subj_id, "status": "success", **log.get("n_epochs_post_autoreject", {})})
    except Exception as e:
        print(f"sub-{subj_id}: FAILED — {e}")
        batch_summary.append({"subject": subj_id, "status": "failed", "error": str(e)})

df_batch_summary = pd.DataFrame(batch_summary)
df_batch_summary.to_csv(derivatives_dir / "batch_preprocessing_summary.csv", index=False)
display(df_batch_summary)

3 of 5 subjects ready for batch preprocessing.
sub-23: outputs already exist, skipping (overwrite=False).
Reading 0 ... 3207199  =      0.000 ...  3207.199 secs...
Reading c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-26\eeg\ica_qc\sub-26_ica.fif ...
Now restoring ICA solution ...
Ready.
Applying ICA to Raw instance
    Transforming to ICA space (59 components)
    Zeroing out 8 ICA components
    Projecting back using 60 PCA components
Dropped 2 epochs: 19, 34
Applying baseline correction (mode: mean)
Applying baseline correction (mode: mean)
sub-26: preprocessing complete. Encoding epochs: 64, Retrieval epochs: 82
sub-17: outputs already exist, skipping (overwrite=False).


,subject,status,encoding,retrieval
0,23,success,59,83
1,26,success,64,82
2,17,success,63,62
